# Tratamento dos dados

Lê os dados brutos da AMREC em `dados/brutos/amrec/` e gera um CSV por tabela do modelo em `dados/tratados/`,
pronto para a carga no banco (`sql/carga.sql`). O modelo está em `docs/modelo.md`.

- **Betha** (10 municípios): um JSON por registro, baixado pelo notebook `coleta.ipynb`.
- **Atende.net** (Treviso e Morro da Fumaça): CSVs exportados manualmente do portal.

Proteção de dados: CPF nunca é gravado completo. Quando a fonte publica o CPF inteiro, ele é mascarado no padrão
`***.XXX.XXX-**`, o mesmo que vários portais já usam; CPFs escritos em campos de texto também são mascarados.
O notebook não exibe nomes nem documentos: as saídas mostram só contagens e conferências.

In [1]:
import json
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = RAIZ / "dados" / "brutos" / "amrec"
SAIDA = RAIZ / "dados" / "tratados"
ANOS = range(2023, 2027)

MUNICIPIOS = pd.DataFrame(
    [
        ("4220000", "Balneário Rincão", "Betha", "balneario_rincao"),
        ("4204251", "Cocal do Sul", "Betha", "cocal_do_sul"),
        ("4204608", "Criciúma", "Betha", "criciuma"),
        ("4205456", "Forquilhinha", "Betha", "forquilhinha"),
        ("4207007", "Içara", "Betha", "icara"),
        ("4209607", "Lauro Müller", "Betha", "lauro_muller"),
        ("4211207", "Morro da Fumaça", "Atende.net", "morro_da_fumaca"),
        ("4211603", "Nova Veneza", "Betha", "nova_veneza"),
        ("4211702", "Orleans", "Betha", "orleans"),
        ("4217600", "Siderópolis", "Betha", "sideropolis"),
        ("4218350", "Treviso", "Atende.net", "treviso"),
        ("4219002", "Urussanga", "Betha", "urussanga"),
    ],
    columns=["ibge", "nome", "fonte", "pasta"],
)
MUNICIPIOS["pasta"] = MUNICIPIOS["ibge"] + "_" + MUNICIPIOS["pasta"]

## 1. Funções de limpeza

- `texto`: normaliza Unicode (NFC) e espaços; texto vazio vira nulo.
- `chave`: forma de comparação, em maiúsculas, sem acentos e sem pontuação.
- `chave_grafia`: compara grafias em que a origem perdeu as letras acentuadas. Alguns portais publicam `Sade`
  no lugar de `Saúde`; descartar as letras não ASCII dos dois lados faz as duas formas coincidirem.
- `documento`: CPF/CNPJ só com dígitos e `*`. CPF completo é mascarado.
- `mascarar_cpfs`: mascara CPFs formatados (`000.000.000-00`) dentro de campos de texto livre.

In [2]:
CPF_EM_TEXTO = re.compile(r"\b\d{3}\.(\d{3}\.\d{3})-\d{2}\b")


def texto(valor):
    if valor is None:
        return None
    limpo = " ".join(unicodedata.normalize("NFC", str(valor)).split())
    return limpo or None


def chave(valor):
    sem_acento = unicodedata.normalize("NFKD", valor or "").encode("ascii", "ignore").decode()
    return " ".join(re.sub(r"[^\w\s]", " ", sem_acento.upper()).split())


def chave_grafia(valor):
    return "".join(c for c in unicodedata.normalize("NFC", valor or "").upper() if c.isascii() and c.isalnum())


def documento(valor):
    digitos = re.sub(r"[^\d*]", "", valor or "")
    if len(digitos) == 11 and digitos.isdigit():
        digitos = "***" + digitos[3:9] + "**"
    return digitos or None


def mascarar_cpfs(valor):
    valor = texto(valor)
    return CPF_EM_TEXTO.sub(r"***.\1-**", valor) if valor else None


def nome(valor):
    valor = texto(valor)
    return valor.upper() if valor else None


def numero(valor):
    return None if valor in (None, "") else float(valor)


def data_iso(valor):
    return date.fromisoformat(valor) if valor else None


def data_br(valor):
    return pd.to_datetime(valor, format="%d/%m/%Y").date() if isinstance(valor, str) and valor else None


def campo(valor):
    """Campo de texto do Atende.net; colunas que só existem em um município chegam como NaN no outro."""
    return valor if isinstance(valor, str) and valor else None


def padronizar(valor, oficiais):
    """Devolve a grafia oficial correspondente, aceitando acentos perdidos na origem."""
    if valor is None:
        return None
    return {chave_grafia(o): o for o in oficiais}[chave_grafia(valor)]


assert chave_grafia("Saúde") == chave_grafia("Sade")
assert documento("123.456.789-01") == "***456789**"
assert documento("12.345.678/****-**") == "12345678******"
assert mascarar_cpfs("pago a 123.456.789-01 ok") == "pago a ***.456.789-** ok"

## 2. Leitura das fontes

### Betha

Cada consulta tem um `.jsonl` por município. Só entram os registros cujo ID ainda está publicado no portal
(lista gravada pela coleta em `<consulta>_ids.json`); os que foram retirados depois de baixados ficam de fora.

In [3]:
def ler_betha(consulta):
    for pasta in MUNICIPIOS.loc[MUNICIPIOS["fonte"] == "Betha", "pasta"]:
        arquivo = BRUTOS / pasta / f"{consulta}.jsonl"
        if not arquivo.exists():
            continue
        publicados = {i for ids in json.loads(arquivo.with_name(f"{consulta}_ids.json").read_text()).values() for i in ids}
        with arquivo.open(encoding="utf-8") as f:
            for linha in f:
                registro = json.loads(linha)
                if registro["_id"] in publicados:
                    yield registro


retirados = {}
for consulta in ["execucao_detalhada", "diarias", "extraorcamentarias"]:
    gravados = sum(1 for p in BRUTOS.glob(f"*/{consulta}.jsonl") for _ in p.open(encoding="utf-8"))
    retirados[consulta] = gravados - sum(1 for _ in ler_betha(consulta))
retirados

{'execucao_detalhada': 3, 'diarias': 1, 'extraorcamentarias': 0}

### Atende.net

Os CSVs vêm em latin-1, separados por `;`, com todo campo entre aspas, e os textos livres têm quebras de linha e
aspas sem escape. Todo registro começa com o nome da entidade entre aspas seguido de `";"`: o arquivo é dividido
nesses pontos e cada registro é separado pelos delimitadores `";"`. A leitura falha se algum registro sair com um
número de campos diferente do cabeçalho.

O número do empenho aparece como `1 / 2025`, `129/2025` ou `824-0/2025` (o sufixo `-0` só existe em Morro da
Fumaça e é sempre zero).

In [4]:
INICIO_DE_REGISTRO = re.compile(r'\n(?="[^"\n;]+";")')
FORMATO_EMPENHO = r"^\s*(\d+)\s*(?:-\s*(\d+)\s*)?/\s*(\d{4})\s*$"


def ler_csv_ipm(arquivo):
    with open(arquivo, encoding="latin-1", newline="") as f:
        conteudo = f.read().replace("\r\n", "\n").strip("\n")
    cabecalho, corpo = conteudo.split("\n", 1)
    colunas = cabecalho[1:-1].split('";"')
    brutos = INICIO_DE_REGISTRO.split(corpo)
    assert all(r.startswith('"') and r.endswith('"') for r in brutos), arquivo.name
    registros = [r[1:-1].split('";"') for r in brutos]
    tamanhos = {len(r) for r in registros}
    assert tamanhos == {len(colunas)}, f"{arquivo.name}: registros com {tamanhos} campos"
    return pd.DataFrame(registros, columns=colunas)


def ler_consulta_ipm(consulta):
    partes = []
    for ibge, pasta in MUNICIPIOS.loc[MUNICIPIOS["fonte"] == "Atende.net", ["ibge", "pasta"]].itertuples(index=False):
        for ano in ANOS:
            partes.append(ler_csv_ipm(BRUTOS / pasta / f"{consulta}_{ano}.csv").assign(ibge=ibge, ano_arquivo=ano))
    df = pd.concat(partes, ignore_index=True)
    partes_empenho = df["Empenho"].str.extract(FORMATO_EMPENHO)
    assert partes_empenho[[0, 2]].notna().all().all(), f"{consulta}: número de empenho fora do formato"
    assert partes_empenho[1].dropna().eq("0").all(), f"{consulta}: sufixo de empenho diferente de -0"
    return df.assign(numero_empenho=partes_empenho[0].astype(int), ano_empenho=partes_empenho[2].astype(int))


ipm = {c: ler_consulta_ipm(c) for c in ["empenho_emitido", "liquidacao", "pagamentos", "diarias"]}
pd.DataFrame({c: df.groupby(["ibge", "ano_arquivo"]).size() for c, df in ipm.items()})

empenho_emitido  liquidacao  pagamentos  diarias
ibge    ano_arquivo                                                  
4211207 2023                   10903       16431       19906      930
        2024                   11337       17251       22521     1060
        2025                   11333       17746       22504     1532
        2026                    9431       14753       18910     1113
4218350 2023                    6753        6783        7998       52
        2024                    6950        7010        8546      114
        2025                    7154        7525        9523      168
        2026                    6755        7082        8569      155

## 3. Cadastros

Credores, materiais e documentos fiscais recebem um ID sequencial na primeira vez em que aparecem.

- **Credor:** identificado por (documento, nome). O documento vem mascarado de formas diferentes em cada
  município, e o mesmo documento mascarado pertence a pessoas diferentes, então ele sozinho não basta.
- **Material:** identificado por (descrição, unidade de medida), normalizadas.
- **Documento fiscal:** identificado por (emitente, tipo, número, data de emissão). Recibos, guias e documentos
  "Outros" reutilizam números, e só a data os separa. O valor não é do documento, e sim do vínculo com o empenho:
  uma nota dividida entre vários empenhos aparece em cada um com a parte que lhe cabe.

In [5]:
TIPOS_EMPENHO = ["Ordinário", "Estimativo", "Global"]
TIPOS_ANULACAO = ["Parcial", "Total"]

credores, materiais, documentos_fiscais = {}, {}, {}
divergencias_documento = Counter()


def id_credor(doc, nome_credor):
    identificacao = (documento(doc), nome(nome_credor))
    if identificacao == (None, None):
        return None
    return credores.setdefault(identificacao, len(credores) + 1)


def id_material(descricao, unidade):
    return materiais.setdefault((nome(descricao), nome(unidade)), len(materiais) + 1)


def id_documento_fiscal(d):
    emitente = id_credor(d.get("cpfCnpjEmitente"), d.get("nomeEmitente"))
    tipo = texto(d.get("tipo"))
    emissao = data_iso(d.get("dataEmissao"))
    identificacao = (emitente, chave_grafia(tipo), texto(str(d.get("numero") or "")), emissao)
    chave_acesso = re.sub(r"\D", "", d.get("chaveAutenticacao") or "") or None
    if identificacao not in documentos_fiscais:
        documentos_fiscais[identificacao] = (len(documentos_fiscais) + 1, emitente, tipo, identificacao[2], emissao,
                                             chave_acesso)
    elif documentos_fiscais[identificacao][5] != chave_acesso:
        divergencias_documento["chave de acesso diferente"] += 1
    return documentos_fiscais[identificacao][0]

## 4. Execução da despesa no Betha

Um registro da execução detalhada é um empenho com as fases da despesa em listas internas. Regras aplicadas:

- **Valor do empenho:** `valorEmpenho`, o valor original. O campo `valorEmpenhado` não tem o mesmo significado em
  todos os municípios (às vezes já desconta as anulações) e vem vazio 8 mil vezes; as anulações ficam na própria
  tabela.
- **Elemento:** dígitos 5 e 6 do código de 14 dígitos, que formam o elemento nacional.
- **Liquidações repetidas:** quando a mesma liquidação aparece duas vezes no empenho com conteúdo idêntico, entra
  uma vez.
- **Retenções:** Içara e Urussanga publicam a retenção na liquidação e de novo no pagamento. Entram as da
  liquidação; das do pagamento, só as que não têm correspondente de mesmo valor na liquidação. Essas se ligam à
  liquidação indicada pelo pagamento ou, se ela não existir, direto ao empenho.
- **Valor anulado** de liquidações e pagamentos: soma das anulações publicadas em cada um.

In [6]:
empenhos, itens, anulacoes, liquidacoes, retencoes, pagamentos = [], [], [], [], [], []
empenho_documentos = Counter()
nomes_elemento = defaultdict(Counter)
elemento_por_descricao = defaultdict(Counter)
liquidacoes_repetidas = 0

for r in ler_betha("execucao_detalhada"):
    id_empenho = len(empenhos) + 1
    codigo_elemento = r["mascaraElemento"][4:6]
    nomes_elemento[codigo_elemento][texto(r["descricaoDetalhamentoElemento"])] += 1
    elemento_por_descricao[chave(r["descricaoElemento"])][codigo_elemento] += 1
    empenhos.append({
        "id_empenho": id_empenho,
        "ibge": r["_ibge"],
        "id_origem": r["_id"],
        "entidade": texto(r["nomeEntidade"]),
        "numero": int(r["numeroEmpenho"]),
        "ano": r["anoExercicio"],
        "data_emissao": date.fromisoformat(r["dataEmpenho"]),
        "tipo": padronizar(texto(r["tipoEmpenho"]), TIPOS_EMPENHO),
        "id_credor": id_credor(r["credor"]["cnpjCpfCredor"], r["credor"]["nomeCredor"]),
        "codigo_elemento": codigo_elemento,
        "elemento_inferido": False,
        "funcao": texto(r["descricaoFuncao"]),
        "historico": mascarar_cpfs(r.get("historicoEmpenho")),
        "valor": r["valorEmpenho"],
    })
    for sequencia, item in enumerate(r.get("itens") or [], 1):
        itens.append((id_empenho, sequencia, id_material(item["descricaoItem"], item.get("unidadeMedidaItem")),
                      numero(item.get("quantidadeItem")), numero(item.get("valorUnitarioItem")),
                      numero(item.get("valorTotalItem"))))
    for d in r.get("documentosFiscais") or []:
        empenho_documentos[(id_empenho, id_documento_fiscal(d))] += numero(d.get("valorDocumento")) or 0
    for sequencia, a in enumerate(r.get("anulacoes") or [], 1):
        anulacoes.append((id_empenho, sequencia, date.fromisoformat(a["dataAnulacao"]),
                          padronizar(texto(a.get("tipoAnulacao")), TIPOS_ANULACAO),
                          mascarar_cpfs(a.get("motivoAnulacao")), a["valorAnulacao"]))

    liquidacao_por_numero, vistas, retido_na_liquidacao = {}, set(), Counter()
    for q in r.get("liquidacoes") or []:
        conteudo = json.dumps(q, sort_keys=True)
        if conteudo in vistas:
            liquidacoes_repetidas += 1
            continue
        vistas.add(conteudo)
        id_liquidacao = len(liquidacoes) + 1
        anulado = sum(a["valorAnulacao"] for a in q.get("anulacoesLiquidacao") or [])
        liquidacoes.append((id_liquidacao, id_empenho, str(q["idLiquidacao"]), date.fromisoformat(q["data"]),
                            q["valor"], round(anulado, 2)))
        liquidacao_por_numero.setdefault(str(q["idLiquidacao"]), id_liquidacao)
        for x in q.get("retencoesLiquidacao") or []:
            retencoes.append((id_liquidacao, None, date.fromisoformat(x["dataRetencao"]), x["valorRetido"]))
            retido_na_liquidacao[round(x["valorRetido"], 2)] += 1

    for p in r.get("pagamentos") or []:
        anulado = sum(a["valorAnulacao"] for a in p.get("anulacoesPagamento") or [])
        pagamentos.append((id_empenho, None, date.fromisoformat(p["dataPagamento"]), p["valorPagamento"],
                           round(anulado, 2), mascarar_cpfs(p.get("historico"))))
        for x in p.get("retencoesPagamento") or []:
            valor = round(x["valorRetencao"], 2)
            if retido_na_liquidacao[valor] > 0:
                retido_na_liquidacao[valor] -= 1
                continue
            id_liquidacao = liquidacao_por_numero.get(str(p.get("idLiquidacao")))
            retencoes.append((id_liquidacao, None if id_liquidacao else id_empenho,
                              data_iso(x.get("dataRetencao")), x["valorRetencao"]))

{"empenhos": len(empenhos), "itens": len(itens), "documentos": len(documentos_fiscais),
 "liquidações": len(liquidacoes), "liquidações repetidas descartadas": liquidacoes_repetidas,
 "retenções": len(retencoes), "pagamentos": len(pagamentos),
 "documentos com chave de acesso divergente": divergencias_documento["chave de acesso diferente"]}

{'empenhos': 389646,
 'itens': 422161,
 'documentos': 384546,
 'liquidações': 452126,
 'liquidações repetidas descartadas': 7,
 'retenções': 251741,
 'pagamentos': 482142,
 'documentos com chave de acesso divergente': 142}

## 5. Diárias e pagamentos extraorçamentários no Betha

O valor unitário da diária não é guardado: em todos os registros ele é exatamente o valor total dividido pela
quantidade. Quando a finalidade da viagem vem vazia, usa-se o histórico.

In [7]:
diarias = []
for r in ler_betha("diarias"):
    diarias.append({
        "ibge": r["_ibge"],
        "id_credor": id_credor(r.get("cnpjCpfCredor"), r.get("credor")),
        "entidade": texto(r.get("nomeEntidade")),
        "cargo": texto(r.get("cargoCredor")),
        "data_inicio": data_iso(r.get("dataInicial")),
        "data_fim": data_iso(r.get("dataFinal")),
        "destino": texto(r.get("localDestino")),
        "finalidade": mascarar_cpfs(r.get("finalidade")) or mascarar_cpfs(r.get("historico")),
        "quantidade": numero(r.get("quantidade")),
        "valor": numero(r.get("valorTotal")),
    })

extraorcamentarios = []
for r in ler_betha("extraorcamentarias"):
    for p in r.get("pagamentos") or []:
        extraorcamentarios.append({
            "ibge": r["_ibge"],
            "conta": texto(r.get("classificacao")),
            "id_credor": id_credor(p.get("cpfCnpjBeneficiario"), p.get("nomeBeneficiario")),
            "data": date.fromisoformat(p["data"]),
            "valor": p["valor"],
            "historico": mascarar_cpfs(p.get("historico")),
        })

{"diárias": len(diarias), "pagamentos extraorçamentários": len(extraorcamentarios)}

{'diárias': 21684, 'pagamentos extraorçamentários': 153235}

## 6. Atende.net

- **Empenho:** chave (município, entidade, número, ano). O tipo vem da espécie informada nas liquidações e
  diárias do mesmo empenho. A função não é publicada.
- **Elemento:** o portal publica só a descrição do subelemento, convertida para o elemento nacional
  (`elemento_inferido = verdadeiro`):
  1. pela correspondência entre descrição e elemento observada no Betha, quando um elemento responde por 95% ou
     mais dos casos;
  2. quando a descrição se divide entre 36 (serviço de pessoa física) e 39 (pessoa jurídica), pelo documento do
     credor: CPF → 36, CNPJ → 39;
  3. pela tabela manual abaixo, para descrições abreviadas sem par no Betha.

  O que não se resolve fica com elemento nulo.
- **Restos a pagar:** liquidações, pagamentos e diárias de empenhos anteriores a 2023 ficam de fora, porque o
  empenho de origem não está no período coletado.
- **Retenção:** pagamentos do tipo "Retenção" vão para a tabela de retenções, ligados ao empenho.
- **Diária:** o credor é o do empenho da diária quando o nome coincide; o portal não publica o documento do
  servidor na consulta de diárias.

In [8]:
ELEMENTO_MANUAL = {
    "MATERIAL PARA MANUTENCAO BENS IMOVEIS": "30",
    "MATERIAL LIMPEZA E PRODUTOS DE HIGIENIZACAO": "30",
    "DIVERSAS INDENIZACOES E RESTITUICOES TRABALHISTAS": "94",
    "GRATIFICACAO DE TEMPO DE SERVICO": "11",
    "OUTROS SERVICOS TERCEIROS PESSOA FISICA PAGTO ANTECIPADO": "36",
    "MANUTENCAO E CONSERVACAO BENS IMOVEIS": "39",
    "SERVICOS MEDICO HOSPITALARES ODONTOLOGICOS E LABORATORIAIS": "39",
    "APARELHOS EQUIP UTENS MEDICO ODONTOLOGICOS LABORATORIAIS E HOSPITALARES": "52",
    "OUTROS EQUIPAMENTOS MATERIAL PERMANENTE": "52",
    "CONTRIBUICAO PREVIDENCIARIA S SERV TERCEIROS PESSOA FISICA": "47",
    "ASSESSORIA E CONSULTORIA TACNICA OU JURIDICA": "35",
}


def elemento_ipm(descricao, doc):
    descricao = chave(descricao)
    if descricao in ELEMENTO_MANUAL:
        return ELEMENTO_MANUAL[descricao]
    contagem = elemento_por_descricao.get(descricao)
    if not contagem:
        return None
    if set(contagem) == {"36", "39"} and doc and len(doc) in (11, 14):
        return "36" if len(doc) == 11 else "39"
    codigo, vezes = contagem.most_common(1)[0]
    return codigo if vezes / contagem.total() >= 0.95 else None


CHAVE_EMPENHO = ["ibge", "Entidade", "numero_empenho", "ano_empenho"]
emp = ipm["empenho_emitido"]
assert not emp.duplicated(CHAVE_EMPENHO).any()
assert (emp["ano_empenho"] == emp["ano_arquivo"]).all()

especie = pd.concat([ipm["liquidacao"], ipm["diarias"]])[CHAVE_EMPENHO + ["Espécie"]].drop_duplicates(CHAVE_EMPENHO)
emp = emp.merge(especie, on=CHAVE_EMPENHO, how="left", validate="one_to_one")
emp["id_empenho"] = range(len(empenhos) + 1, len(empenhos) + len(emp) + 1)

for e in emp.itertuples(index=False):
    doc = documento(getattr(e, "_2"))
    empenhos.append({
        "id_empenho": e.id_empenho,
        "ibge": e.ibge,
        "id_origem": f"{e.Entidade}|{e.numero_empenho}/{e.ano_empenho}",
        "entidade": texto(e.Entidade),
        "numero": e.numero_empenho,
        "ano": e.ano_empenho,
        "data_emissao": pd.to_datetime(getattr(e, "Emissão"), format="%d/%m/%Y").date(),
        "tipo": padronizar(campo(getattr(e, "Espécie")), TIPOS_EMPENHO),
        "id_credor": id_credor(getattr(e, "_2"), getattr(e, "_1")),
        "codigo_elemento": elemento_ipm(getattr(e, "_5"), doc),
        "elemento_inferido": True,
        "funcao": None,
        "historico": mascarar_cpfs(getattr(e, "_6")),
        "valor": float(getattr(e, "_11")),
    })

Os nomes de coluna do Atende.net têm acentos e espaços, que o `itertuples` troca por posições (`_1`, `_2`...).
A célula abaixo confere que cada posição usada acima corresponde à coluna esperada.

In [9]:
colunas = list(ipm["empenho_emitido"].columns)
assert [colunas[i] for i in (1, 2, 5, 6, 11)] == [
    "Credor - Nome/Razão", "Credor - CPF/CNPJ", "Descrição Despesa", "Histórico/Objeto", "Valor - Empenhado"]

In [10]:
id_empenho_ipm = emp.set_index(CHAVE_EMPENHO)["id_empenho"]
restos = {}


def ligar(df, consulta):
    ligado = df.join(id_empenho_ipm, on=CHAVE_EMPENHO)
    sem_empenho = ligado["id_empenho"].isna()
    assert (ligado.loc[sem_empenho, "ano_empenho"] < ANOS[0]).all(), f"{consulta}: registro sem empenho no período"
    restos[consulta] = int(sem_empenho.sum())
    return ligado[~sem_empenho].astype({"id_empenho": int})


for q in ligar(ipm["liquidacao"], "liquidacao").itertuples(index=False):
    liquidacoes.append((len(liquidacoes) + 1, q.id_empenho, getattr(q, "Liquidação"),
                        pd.to_datetime(q.Data, format="%d/%m/%Y").date(), float(getattr(q, "_5")), None))

pag = ligar(ipm["pagamentos"], "pagamentos")
for p in pag.itertuples(index=False):
    data_pagamento = pd.to_datetime(p.Data, format="%d/%m/%Y").date()
    if getattr(p, "_10") == "Retenção":
        retencoes.append((None, p.id_empenho, data_pagamento, float(getattr(p, "_11"))))
    else:
        pagamentos.append((p.id_empenho, texto(getattr(p, "_3")), data_pagamento, float(getattr(p, "_11")), None,
                           mascarar_cpfs(getattr(p, "Histórico"))))

credor_do_empenho = {e["id_empenho"]: e["id_credor"] for e in empenhos[len(empenhos) - len(emp):]}
nome_do_credor = {i: n for (_, n), i in credores.items()}
dia = ligar(ipm["diarias"], "diarias")
for d in dia.to_dict("records"):
    credor = credor_do_empenho[d["id_empenho"]]
    if nome_do_credor.get(credor) != nome(d["Credor"]):
        credor = id_credor(None, d["Credor"])
    diarias.append({
        "ibge": d["ibge"],
        "id_credor": credor,
        "entidade": texto(d["Entidade"]),
        "cargo": texto(d["Cargo"]),
        "data_inicio": data_br(d.get("Data de Início")),
        "data_fim": data_br(d.get("Data do Fim")),
        "destino": texto(campo(d.get("Destino"))),
        "finalidade": mascarar_cpfs(campo(d.get("Objetivo"))),
        "quantidade": numero(campo(d.get("Número de Diárias"))),
        "valor": float(d["Valor Diárias"]),
    })

colunas = list(ipm["pagamentos"].columns)
assert [colunas[i] for i in (3, 5, 10, 11)] == ["N° Ordem de Pagamento", "Data", "Tipo Pagamento", "Valor Pago"]
assert list(ipm["liquidacao"].columns)[5] == "Valor Liquidação"
restos

{'liquidacao': 170, 'pagamentos': 330, 'diarias': 0}

## 7. Montagem das tabelas

In [11]:
FUNCOES = {
    1: "Legislativa", 2: "Judiciária", 3: "Essencial à Justiça", 4: "Administração", 5: "Defesa Nacional",
    6: "Segurança Pública", 7: "Relações Exteriores", 8: "Assistência Social", 9: "Previdência Social",
    10: "Saúde", 11: "Trabalho", 12: "Educação", 13: "Cultura", 14: "Direitos da Cidadania", 15: "Urbanismo",
    16: "Habitação", 17: "Saneamento", 18: "Gestão Ambiental", 19: "Ciência e Tecnologia", 20: "Agricultura",
    21: "Organização Agrária", 22: "Indústria", 23: "Comércio e Serviços", 24: "Comunicações", 25: "Energia",
    26: "Transporte", 27: "Desporto e Lazer", 28: "Encargos Especiais",
}
codigo_funcao = {chave_grafia(n): c for c, n in FUNCOES.items()}

coleta = {}
for ibge, fonte, pasta in MUNICIPIOS[["ibge", "fonte", "pasta"]].itertuples(index=False):
    if fonte == "Betha":
        coleta[ibge] = date.fromisoformat(json.loads((BRUTOS / pasta / "manifesto.json").read_text(encoding="utf-8"))["fim"][:10])
    else:
        coleta[ibge] = date.fromtimestamp(max(p.stat().st_mtime for p in (BRUTOS / pasta).glob("*.csv")))

tabelas = {}
tabelas["municipio"] = MUNICIPIOS.assign(data_coleta=MUNICIPIOS["ibge"].map(coleta))[["ibge", "nome", "fonte", "data_coleta"]]
tabelas["funcao"] = pd.DataFrame(list(FUNCOES.items()), columns=["codigo", "nome"])
tabelas["elemento_despesa"] = pd.DataFrame(
    [(c, nomes.most_common(1)[0][0]) for c, nomes in sorted(nomes_elemento.items())], columns=["codigo", "nome"])
tabelas["credor"] = pd.DataFrame([(i, d, n) for (d, n), i in credores.items()], columns=["id_credor", "documento", "nome"])
tabelas["material"] = pd.DataFrame([(i, d, u) for (d, u), i in materiais.items()],
                                   columns=["id_material", "descricao", "unidade_medida"])
tabelas["documento_fiscal"] = pd.DataFrame(
    [v for v in documentos_fiscais.values()],
    columns=["id_documento", "id_emitente", "tipo", "numero", "data_emissao", "chave_acesso"])

empenho = pd.DataFrame(empenhos)
empenho["codigo_funcao"] = empenho["funcao"].map(lambda f: codigo_funcao[chave_grafia(f)] if isinstance(f, str) else None).astype("Int64")
tabelas["empenho"] = empenho.drop(columns="funcao")[[
    "id_empenho", "ibge", "id_origem", "entidade", "numero", "ano", "data_emissao", "tipo", "id_credor",
    "codigo_elemento", "elemento_inferido", "codigo_funcao", "historico", "valor"]]
tabelas["item_empenho"] = pd.DataFrame(itens, columns=[
    "id_empenho", "sequencia", "id_material", "quantidade", "valor_unitario", "valor_total"])
tabelas["empenho_documento_fiscal"] = pd.DataFrame(
    [(e, d, round(v, 2)) for (e, d), v in sorted(empenho_documentos.items())],
    columns=["id_empenho", "id_documento", "valor"])
tabelas["anulacao_empenho"] = pd.DataFrame(anulacoes, columns=[
    "id_empenho", "sequencia", "data", "tipo", "motivo", "valor"])
tabelas["liquidacao"] = pd.DataFrame(liquidacoes, columns=[
    "id_liquidacao", "id_empenho", "numero", "data", "valor", "valor_anulado"])
tabelas["retencao"] = pd.DataFrame(retencoes, columns=["id_liquidacao", "id_empenho", "data", "valor"])
tabelas["retencao"].insert(0, "id_retencao", range(1, len(retencoes) + 1))
tabelas["pagamento"] = pd.DataFrame(pagamentos, columns=[
    "id_empenho", "numero_ordem", "data", "valor", "valor_anulado", "historico"])
tabelas["pagamento"].insert(0, "id_pagamento", range(1, len(pagamentos) + 1))
tabelas["diaria"] = pd.DataFrame(diarias)
tabelas["diaria"].insert(0, "id_diaria", range(1, len(diarias) + 1))
tabelas["pagamento_extraorcamentario"] = pd.DataFrame(extraorcamentarios)
tabelas["pagamento_extraorcamentario"].insert(0, "id_pagamento_extra", range(1, len(extraorcamentarios) + 1))

for nome_tabela in ["retencao"]:
    for coluna in ["id_liquidacao", "id_empenho"]:
        tabelas[nome_tabela][coluna] = tabelas[nome_tabela][coluna].astype("Int64")
for nome_tabela in ["empenho", "diaria", "pagamento_extraorcamentario", "documento_fiscal"]:
    coluna = "id_emitente" if nome_tabela == "documento_fiscal" else "id_credor"
    tabelas[nome_tabela][coluna] = tabelas[nome_tabela][coluna].astype("Int64")

## 8. Conferências

Antes de exportar: chaves primárias únicas e sem nulos, chaves estrangeiras sem órfãos, regras de domínio e o
critério de volume do trabalho.

In [12]:
CHAVES_PRIMARIAS = {
    "municipio": ["ibge"], "funcao": ["codigo"], "elemento_despesa": ["codigo"], "credor": ["id_credor"],
    "material": ["id_material"], "documento_fiscal": ["id_documento"], "empenho": ["id_empenho"],
    "item_empenho": ["id_empenho", "sequencia"], "empenho_documento_fiscal": ["id_empenho", "id_documento"],
    "anulacao_empenho": ["id_empenho", "sequencia"], "liquidacao": ["id_liquidacao"],
    "retencao": ["id_retencao"], "pagamento": ["id_pagamento"], "diaria": ["id_diaria"],
    "pagamento_extraorcamentario": ["id_pagamento_extra"],
}
CHAVES_UNICAS = {
    "municipio": ["nome"], "funcao": ["nome"], "credor": ["documento", "nome"],
    "material": ["descricao", "unidade_medida"],
    "documento_fiscal": ["id_emitente", "tipo", "numero", "data_emissao"],
    "empenho": ["ibge", "id_origem"],
}
CHAVES_ESTRANGEIRAS = [
    ("documento_fiscal", "id_emitente", "credor", "id_credor"),
    ("empenho", "ibge", "municipio", "ibge"),
    ("empenho", "id_credor", "credor", "id_credor"),
    ("empenho", "codigo_elemento", "elemento_despesa", "codigo"),
    ("empenho", "codigo_funcao", "funcao", "codigo"),
    ("item_empenho", "id_empenho", "empenho", "id_empenho"),
    ("item_empenho", "id_material", "material", "id_material"),
    ("empenho_documento_fiscal", "id_empenho", "empenho", "id_empenho"),
    ("empenho_documento_fiscal", "id_documento", "documento_fiscal", "id_documento"),
    ("anulacao_empenho", "id_empenho", "empenho", "id_empenho"),
    ("liquidacao", "id_empenho", "empenho", "id_empenho"),
    ("retencao", "id_liquidacao", "liquidacao", "id_liquidacao"),
    ("retencao", "id_empenho", "empenho", "id_empenho"),
    ("pagamento", "id_empenho", "empenho", "id_empenho"),
    ("diaria", "ibge", "municipio", "ibge"),
    ("diaria", "id_credor", "credor", "id_credor"),
    ("pagamento_extraorcamentario", "ibge", "municipio", "ibge"),
    ("pagamento_extraorcamentario", "id_credor", "credor", "id_credor"),
]

for t, pk in CHAVES_PRIMARIAS.items():
    assert tabelas[t][pk].notna().all().all(), f"{t}: chave primária nula"
    assert not tabelas[t].duplicated(pk).any(), f"{t}: chave primária repetida"
for t, uk in CHAVES_UNICAS.items():
    assert not tabelas[t].duplicated(uk).any(), f"{t}: chave única {uk} repetida"
orfaos = {}
for filha, coluna, pai, chave_pai in CHAVES_ESTRANGEIRAS:
    valores = tabelas[filha][coluna].dropna()
    orfaos[f"{filha}.{coluna}"] = int((~valores.isin(tabelas[pai][chave_pai])).sum())
assert not any(orfaos.values()), {k: v for k, v in orfaos.items() if v}

r = tabelas["retencao"]
assert (r["id_liquidacao"].isna() != r["id_empenho"].isna()).all(), "retenção sem exatamente uma ligação"
assert tabelas["empenho"]["tipo"].dropna().isin(TIPOS_EMPENHO).all()
assert tabelas["anulacao_empenho"]["tipo"].isin(TIPOS_ANULACAO).all()
assert tabelas["empenho"]["ano"].isin(list(ANOS)).all()
assert tabelas["credor"]["documento"].dropna().str.fullmatch(r"[\d*]{11}|[\d*]{14}").all(), "documento fora do formato"
assert not tabelas["credor"]["documento"].dropna().str.fullmatch(r"\d{11}").any(), "CPF completo"

contagem = pd.Series({t: len(df) for t, df in tabelas.items()}, name="linhas").sort_values(ascending=False)
assert (contagem >= 10_000).mean() >= 0.8, "menos de 80% das tabelas com 10 mil linhas"
contagem.to_frame().assign(acima_de_10_mil=contagem >= 10_000)

,linhas,acima_de_10_mil
pagamento,576524,True
liquidacao,546537,True
empenho,460262,True
empenho_documento_fiscal,445096,True
item_empenho,422161,True
documento_fiscal,384546,True
retencao,275506,True
pagamento_extraorcamentario,153235,True
material,84996,True
anulacao_empenho,47020,True


Qualidade dos campos derivados de regras: quantos empenhos do Atende.net ficaram sem elemento, e quanto do
volume financeiro isso representa.

In [13]:
e = tabelas["empenho"]
ipm_emp = e[e["elemento_inferido"]]
pd.DataFrame({
    "empenhos": ipm_emp.groupby("ibge").size(),
    "sem elemento": ipm_emp[ipm_emp["codigo_elemento"].isna()].groupby("ibge").size(),
    "% do valor sem elemento": (ipm_emp[ipm_emp["codigo_elemento"].isna()].groupby("ibge")["valor"].sum()
                                / ipm_emp.groupby("ibge")["valor"].sum() * 100).round(1),
}).fillna(0)

,empenhos,sem elemento,% do valor sem elemento
ibge,,,
4211207,43004,1126,2.5
4218350,27612,317,1.9


## 9. Exportação

Um CSV em UTF-8 por tabela, com cabeçalho, em `dados/tratados/` (fora do Git).

In [14]:
SAIDA.mkdir(parents=True, exist_ok=True)
for t, df in tabelas.items():
    df.to_csv(SAIDA / f"{t}.csv", index=False, encoding="utf-8")

pd.DataFrame({
    "linhas": contagem,
    "MB": pd.Series({t: round((SAIDA / f"{t}.csv").stat().st_size / 2**20, 1) for t in tabelas}),
})

,linhas,MB
anulacao_empenho,47020,3.2
credor,28814,1.4
diaria,26808,8.7
documento_fiscal,384546,22.4
elemento_despesa,50,0.0
empenho,460262,157.9
empenho_documento_fiscal,445096,9.2
funcao,28,0.0
item_empenho,422161,13.2
liquidacao,546537,21.5
